# DFE resampler periodic-frame test

The 100 MHz harness records output-valid cycles across precisely one period of accepted inputs after two warm-up periods. Targets are exact count and FFT bin, passband gain within 0.15 dB of the AMD bit-accurate model, and at least 55 dB image or alias suppression. Compare the circularly aligned impulse to the model within 2 output codes. The two instances use 2/3 (3 input lanes) and 2× (1 input lane), both with 2 output lanes. The model vectors were generated before board measurement.

In [ ]:
import json, os, sys
from pathlib import Path
import numpy as np
from pynq import Overlay
import pynq_dfe
import matplotlib.pyplot as plt

root = Path(os.environ.get("DFE_REPO", "..")).resolve()
example = root / "packages/pynq-dfe/examples/resampler"
sys.path[:0] = [str(example), str(root / "overlay/dfe-resampler-nlf")]
from capture import capture
from model import *
bit = Path(os.environ["DFE_BITFILE"])
out = Path(os.environ.get("DFE_OUTPUT", ".")).resolve()
out.mkdir(exist_ok=True)
ol = Overlay(str(bit))
rows, traces = [], {}

def check(name, measured, predicted, passed):
    rows.append(dict(test=name, measured=float(measured), predicted=float(predicted), passed=bool(passed)))

In [ ]:
golden = np.load(example / "golden.npz")
for mode, inst in enumerate(("dec", "intp")):
    ip = pynq_dfe.ResamplerIP.from_hwh(bit.with_suffix(".hwh"), inst)
    for kind in (("impulse", "tone", "alias") if mode == 0 else ("impulse", "tone")):
        x = stimulus(ip.input_lanes, kind)
        y, beats, faults = capture(ol, mode, x)
        key = inst + "_" + kind
        traces[key] = y
        ref = golden[key]
        count = beats * ip.output_lanes
        expected = len(x) * ip.ratio
        check(key+" samples", count, expected, count == expected)
        check(key+" stream faults", faults, 0, faults == 0)
        s, g = spectrum(y), spectrum(ref)
        if kind == "impulse":
            lag = int(np.argmax(abs(y))) - int(np.argmax(abs(ref)))
            err = np.max(np.abs(np.roll(y, -lag) - ref))
            check(key+" max code error", err, 0, err <= 2)
        if kind == "tone":
            k = np.argmax(s)
            check(key+" peak bin", k, TONE_BIN, k == TONE_BIN)
            gain, target = db(s[TONE_BIN]/AMPLITUDE), db(g[TONE_BIN]/AMPLITUDE)
            check(key+" gain dB", gain, target, abs(gain-target) <= GAIN_TOL_DB)
        if kind == "alias" or (mode == 1 and kind == "tone"):
            k = TONE_BIN + BEATS
            a, b = (max(s), max(g)) if kind == "alias" else (s[k], g[k])
            rejection, target = -db(a/AMPLITUDE), -db(b/AMPLITUDE)
            check(key+" rejection dB", rejection, target, rejection >= MIN_REJECTION_DB)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 5))
for i, inst in enumerate(("dec", "intp")):
    y = traces[inst+"_impulse"]
    axes[i, 0].plot(np.arange(-48,49), np.roll(y.real, 48-np.argmax(abs(y)))[:97])
    axes[i, 0].set(xlabel="Output sample relative to peak", ylabel=inst+" impulse (codes)")
    axes[i, 1].plot(np.fft.fftshift(np.fft.fftfreq(2048)), np.fft.fftshift(spectrum(traces[inst+"_tone"])))
    axes[i, 1].set(xlabel="Cycles per output sample", ylabel="Amplitude (codes)")
fig.tight_layout()
fig.savefig(out / "response.png")

In [ ]:
np.savez_compressed(out / "captures.npz", **traces)
(out / "measurements.json").write_text(json.dumps(rows, indent=2))
for r in rows:
    print(f"{r['test']:38s} {r['measured']:12.6g} {r['predicted']:12.6g}  {'PASS' if r['passed'] else 'FAIL'}")
assert all(r["passed"] for r in rows)